# Sets in Python: Unique Collections and Set Operations

**Day 4 | Python in 30 Days | Beginner**

---

## What this notebook covers

- What a set is and when to use one
- Creating sets (and the empty-set trap)
- Sets are unordered: no indexing, how to loop and sort
- Fast membership testing
- Adding and removing: `add`, `update`, `remove`, `discard`, `pop`, `clear`
- Set operations: union, intersection, difference, symmetric difference
- Operators vs methods, and the in-place versions (`|=`, `&=`, `-=`, `^=`)
- Subset, superset and disjoint checks
- `frozenset`: the immutable set
- Set comprehensions
- Practical patterns: removing duplicates, finding duplicates, blocklists, skill matching
- Common mistakes, a speed test, and three mini-challenges

## Prerequisites

Lists, tuples and dictionaries from earlier in Day 4, plus `for` loops.

## How to use this notebook

Run the cells from top to bottom (Shift + Enter). Outputs are already saved, so you can read first and then run them yourself to confirm. Whenever a set of strings is printed, we use `sorted()` so your output matches the one shown. Plain sets of strings can print in a different order on every run.

## 1. What is a set?

A **set** is a collection of **unique** items with **no guaranteed order**.

Think of the member list of a WhatsApp group: a person is either in the group or not. Adding the same person twice changes nothing, and the group has no "first" or "second" member in any meaningful sense.

| Property | Set |
|----------|-----|
| Duplicates | Not allowed (they disappear automatically) |
| Order | Not guaranteed, so no indexing or slicing |
| Changeable | Yes (you can add and remove items) |
| Items must be | Immutable (hashable): numbers, strings, tuples are fine; lists and dicts are not |
| Membership test (`in`) | Very fast |

Use a set when the question is *"is it in there?"*, *"what is unique?"*, or *"what do these two groups have in common?"*

## 2. Creating sets

Curly braces with items, or `set()` on any iterable (list, string, range, tuple...).

In [1]:
# Curly braces with items
mumbai_visitors = {"Priya", "Arjun", "Riya"}
print("Type:", type(mumbai_visitors))
print("Members:", sorted(mumbai_visitors))

# Duplicates vanish automatically
chai_orders = {"masala chai", "cutting chai", "masala chai", "ginger chai", "cutting chai"}
print("Unique chai types:", len(chai_orders))
print(sorted(chai_orders))

Type: <class 'set'>
Members: ['Arjun', 'Priya', 'Riya']
Unique chai types: 3
['cutting chai', 'ginger chai', 'masala chai']


In [2]:
# set() converts any iterable
cities = ["Mumbai", "Delhi", "Mumbai", "Pune", "Delhi"]
print("From a list:  ", sorted(set(cities)))
print("From a string:", sorted(set("banana")))      # unique letters
print("From a range: ", set(range(1, 6)))
print("From a tuple: ", sorted(set(("red", "green", "red"))))

From a list:   ['Delhi', 'Mumbai', 'Pune']
From a string: ['a', 'b', 'n']
From a range:  {1, 2, 3, 4, 5}
From a tuple:  ['green', 'red']


### The empty-set trap

`{}` creates an empty **dictionary**, not an empty set. To make an empty set, use `set()`.

In [3]:
not_a_set = {}
real_empty_set = set()
print(type(not_a_set))
print(type(real_empty_set))
print(real_empty_set)

<class 'dict'>
<class 'set'>
set()


### Set items must be immutable (hashable)

Strings, numbers and tuples can go inside a set. Lists and dictionaries cannot, because they can change after being added.

In [4]:
# A tuple works (for example GPS coordinates)
locations = {(19.0760, 72.8777), (28.6139, 77.2090), (19.0760, 72.8777)}
print("Unique locations:", len(locations))

# A list does not
try:
    broken = {["Mumbai", "Pune"], ["Delhi", "Agra"]}
except TypeError as error:
    print("TypeError:", error)

Unique locations: 2
TypeError: unhashable type: 'list'


## 3. Sets are unordered: no indexing, but you can loop

There is no "first item" in a set, so `my_set[0]` is an error. You can loop over a set, and use `sorted()` when you need a predictable order (it returns a **list**).

In [5]:
subjects = {"Physics", "Chemistry", "Math"}

try:
    print(subjects[0])
except TypeError as error:
    print("TypeError:", error)

# Looping over a sorted copy gives a stable order
for subject in sorted(subjects):
    print("-", subject)

print(sorted(subjects, reverse=True))

TypeError: 'set' object is not subscriptable
- Chemistry
- Math
- Physics
['Physics', 'Math', 'Chemistry']


## 4. Membership testing and length

`in` and `not in` work on sets, and they are very fast even for huge sets (we measure this in section 12).

In [6]:
allowed_users = {"priya", "arjun", "riya", "sanjay"}

print("priya" in allowed_users)
print("mallory" in allowed_users)
print("mallory" not in allowed_users)
print("Total allowed:", len(allowed_users))

True
False
True
Total allowed: 4


## 5. Adding and removing items

| Method | What it does | If the item is missing / present |
|--------|--------------|----------------------------------|
| `add(x)` | Add one item | Adding an existing item does nothing |
| `update(iterable, ...)` | Add many items | Existing items are ignored |
| `remove(x)` | Remove an item | Raises `KeyError` if absent |
| `discard(x)` | Remove an item | Does nothing if absent (safe) |
| `pop()` | Remove and return an **arbitrary** item | Raises `KeyError` if the set is empty |
| `clear()` | Remove everything | |

In [7]:
batch = {"Priya", "Arjun"}

batch.add("Riya")
batch.add("Priya")                         # already present: no change
print("After add:", sorted(batch))

batch.update(["Sanjay", "Kavita"])          # many items from a list
batch.update({"Meera"}, ["Vikram"])         # update accepts several iterables
print("After update:", sorted(batch))

batch.remove("Arjun")
print("After remove:", sorted(batch))

After add: ['Arjun', 'Priya', 'Riya']
After update: ['Arjun', 'Kavita', 'Meera', 'Priya', 'Riya', 'Sanjay', 'Vikram']
After remove: ['Kavita', 'Meera', 'Priya', 'Riya', 'Sanjay', 'Vikram']


In [8]:
batch = {"Priya", "Riya", "Sanjay", "Kavita", "Meera", "Vikram"}

# remove() is strict, discard() is forgiving
try:
    batch.remove("Zoya")
except KeyError as error:
    print("KeyError from remove:", error)

batch.discard("Zoya")                       # no error
print("discard('Zoya') did nothing:", len(batch))

# pop() removes an arbitrary item: never rely on which one
removed = batch.pop()
print("pop removed one member:", removed in {"Priya", "Riya", "Sanjay", "Kavita", "Meera", "Vikram"})
print("Remaining:", len(batch))

batch.clear()
print("After clear:", batch)

KeyError from remove: 'Zoya'
discard('Zoya') did nothing: 6
pop removed one member: True
Remaining: 5
After clear: set()


## 6. Set operations

This is where sets shine. Two course batches:

In [9]:
python_batch = {"Priya", "Arjun", "Riya", "Sanjay", "Meera"}
sql_batch    = {"Riya", "Sanjay", "Kavita", "Vikram", "Meera"}
print("Python:", sorted(python_batch))
print("SQL:   ", sorted(sql_batch))

Python: ['Arjun', 'Meera', 'Priya', 'Riya', 'Sanjay']
SQL:    ['Kavita', 'Meera', 'Riya', 'Sanjay', 'Vikram']


| Operation | Operator | Method | Meaning |
|-----------|----------|--------|---------|
| Union | `a \| b` | `a.union(b)` | In either set (or both) |
| Intersection | `a & b` | `a.intersection(b)` | In both sets |
| Difference | `a - b` | `a.difference(b)` | In `a` but not in `b` |
| Symmetric difference | `a ^ b` | `a.symmetric_difference(b)` | In exactly one of the two |

In [10]:
print("Union (either batch):       ", sorted(python_batch | sql_batch))
print("Intersection (both batches):", sorted(python_batch & sql_batch))
print("Python only (a - b):        ", sorted(python_batch - sql_batch))
print("SQL only (b - a):           ", sorted(sql_batch - python_batch))
print("Exactly one batch (a ^ b):  ", sorted(python_batch ^ sql_batch))

Union (either batch):        ['Arjun', 'Kavita', 'Meera', 'Priya', 'Riya', 'Sanjay', 'Vikram']
Intersection (both batches): ['Meera', 'Riya', 'Sanjay']
Python only (a - b):         ['Arjun', 'Priya']
SQL only (b - a):            ['Kavita', 'Vikram']
Exactly one batch (a ^ b):   ['Arjun', 'Kavita', 'Priya', 'Vikram']


Notice that **difference is not symmetric**: `a - b` and `b - a` give different answers. Symmetric difference (`^`) is the "one or the other, but not both" version.

### Operators vs methods

The methods accept **any iterable** and **several arguments** at once. The operators need sets on both sides.

In [11]:
excel_batch = {"Meera", "Zoya"}

# Methods take several arguments
print("In all three batches:", sorted(python_batch.intersection(sql_batch, excel_batch)))
print("Everyone enrolled:   ", sorted(python_batch.union(sql_batch, excel_batch)))

# Methods accept a list, operators do not
print("Union with a list:", sorted(python_batch.union(["Zoya", "Isha"])))
try:
    python_batch | ["Zoya"]
except TypeError as error:
    print("TypeError:", error)

In all three batches: ['Meera']
Everyone enrolled:    ['Arjun', 'Kavita', 'Meera', 'Priya', 'Riya', 'Sanjay', 'Vikram', 'Zoya']
Union with a list: ['Arjun', 'Isha', 'Meera', 'Priya', 'Riya', 'Sanjay', 'Zoya']
TypeError: unsupported operand type(s) for |: 'set' and 'list'


## 7. Subset, superset and disjoint

| Check | Operator | Method | True when |
|-------|----------|--------|-----------|
| Subset | `a <= b` | `a.issubset(b)` | Every item of `a` is in `b` |
| Proper subset | `a < b` | | Subset, and `b` has extra items |
| Superset | `a >= b` | `a.issuperset(b)` | `a` contains every item of `b` |
| Disjoint | | `a.isdisjoint(b)` | No items in common |

In [12]:
core_team = {"Priya", "Arjun"}

print("core_team is a subset of python_batch:  ", core_team.issubset(python_batch))
print("python_batch is a superset of core_team:", python_batch.issuperset(core_team))
print("core_team <= python_batch:", core_team <= python_batch)
print("core_team < core_team (proper subset):", core_team < core_team)

print("{'Zoya'} shares nothing with python_batch:", {"Zoya"}.isdisjoint(python_batch))
print("python_batch and sql_batch are disjoint: ", python_batch.isdisjoint(sql_batch))

core_team is a subset of python_batch:   True
python_batch is a superset of core_team: True
core_team <= python_batch: True
core_team < core_team (proper subset): False
{'Zoya'} shares nothing with python_batch: True
python_batch and sql_batch are disjoint:  False


## 8. In-place operations

`a |= b`, `a &= b`, `a -= b`, `a ^= b` change `a` directly instead of creating a new set. The matching methods are `update`, `intersection_update`, `difference_update` and `symmetric_difference_update`.

In [13]:
staff = {"Priya", "Arjun"}
print("Start:           ", sorted(staff))

staff |= {"Riya"}                       # add members
print("After |= Riya:   ", sorted(staff))

staff &= {"Priya", "Riya", "Zoya"}      # keep only these
print("After &= (keep): ", sorted(staff))

staff -= {"Priya"}                      # remove members
print("After -= Priya:  ", sorted(staff))

staff ^= {"Riya", "Kavita"}             # toggle: Riya leaves, Kavita joins
print("After ^= toggle: ", sorted(staff))

Start:            ['Arjun', 'Priya']
After |= Riya:    ['Arjun', 'Priya', 'Riya']
After &= (keep):  ['Priya', 'Riya']
After -= Priya:   ['Riya']
After ^= toggle:  ['Kavita']


## 9. frozenset: an immutable set

A **frozenset** is a set that cannot change after creation. Because it is hashable, a frozenset can be used as a **dictionary key** or placed inside another set, which normal sets cannot.

In [14]:
weekend = frozenset({"Saturday", "Sunday"})
try:
    weekend.add("Monday")
except AttributeError as error:
    print("AttributeError:", error)

# Order of elements does not matter when comparing or looking up
distance_km = {
    frozenset({"Mumbai", "Pune"}): 150,
    frozenset({"Delhi", "Agra"}): 230,
}
print(distance_km[frozenset({"Pune", "Mumbai"})])   # same pair, reversed
print(distance_km[frozenset({"Agra", "Delhi"})])

AttributeError: 'frozenset' object has no attribute 'add'
150
230


## 10. Set comprehensions

Same idea as list comprehensions, with curly braces: `{expression for item in iterable if condition}`. Duplicates collapse automatically.

In [15]:
# Squares of -3..3: negative and positive give the same square, so duplicates collapse
squares = {n ** 2 for n in range(-3, 4)}
print(sorted(squares))

# First letters of names
names = ["Priya", "Arjun", "Riya", "Anita", "Rahul", "Pooja"]
initials = {name[0] for name in names}
print(sorted(initials))

# With a filter condition
long_names = {name for name in names if len(name) > 4}
print(sorted(long_names))

[0, 1, 4, 9]
['A', 'P', 'R']
['Anita', 'Arjun', 'Pooja', 'Priya', 'Rahul']


## 11. Practical patterns

### Removing duplicates (and keeping the order)

`set(my_list)` removes duplicates but **loses the original order**. When order matters, use `dict.fromkeys()`, which keeps first-seen order.

In [16]:
roll_numbers = [105, 102, 105, 101, 102, 108, 101]

print("With set (order lost):  ", sorted(set(roll_numbers)))
print("Order kept, via dict:   ", list(dict.fromkeys(roll_numbers)))

With set (order lost):   [101, 102, 105, 108]
Order kept, via dict:    [105, 102, 101, 108]


### Finding the duplicates

Track what you have already seen in one set, and collect repeats in another.

In [17]:
seen = set()
duplicates = set()
for roll in roll_numbers:
    if roll in seen:
        duplicates.add(roll)
    else:
        seen.add(roll)

print("Seen once or more:", sorted(seen))
print("Appeared again:   ", sorted(duplicates))

Seen once or more: [101, 102, 105, 108]
Appeared again:    [101, 102, 105]


## 12. Why sets are fast: a quick test

Looking for a value in a **list** checks items one by one. Looking in a **set** jumps straight to the answer. We time 200 lookups of the very last number, the worst case for the list.

In [18]:
import time

big_list = list(range(200_000))
big_set = set(big_list)
target = 199_999

start = time.perf_counter()
for _ in range(200):
    target in big_list
list_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(200):
    target in big_set
set_time = time.perf_counter() - start

print("Set lookup was faster than list lookup:", set_time < list_time)
print("Set was at least 100x faster:", list_time > 100 * set_time)

Set lookup was faster than list lookup: True
Set was at least 100x faster: True


Exact times differ on every computer, but the set wins by a huge margin. For small collections it does not matter. For large ones (user lists, pincode lists, word lists) it does.

---
## Guided examples

### Example 1: Zomato orders (who ordered from which restaurant?)

In [19]:
orders = [
    ("Priya", "Biryani House"), ("Arjun", "Dosa Corner"), ("Priya", "Dosa Corner"),
    ("Riya", "Biryani House"),  ("Arjun", "Biryani House"), ("Priya", "Biryani House"),
]

restaurants = {restaurant for _, restaurant in orders}
biryani_fans = {customer for customer, restaurant in orders if restaurant == "Biryani House"}
dosa_fans = {customer for customer, restaurant in orders if restaurant == "Dosa Corner"}

print("Restaurants used:     ", sorted(restaurants))
print("Ordered from both:    ", sorted(biryani_fans & dosa_fans))
print("Biryani House only:   ", sorted(biryani_fans - dosa_fans))
print("Ordered from either:  ", sorted(biryani_fans | dosa_fans))

Restaurants used:      ['Biryani House', 'Dosa Corner']
Ordered from both:     ['Arjun', 'Priya']
Biryani House only:    ['Riya']
Ordered from either:   ['Arjun', 'Priya', 'Riya']


### Example 2: Attendance across three days

In [20]:
class_roll = {"Priya", "Arjun", "Riya", "Sanjay", "Meera", "Kavita"}
day1 = {"Priya", "Arjun", "Riya", "Sanjay", "Meera"}
day2 = {"Priya", "Riya", "Sanjay", "Arjun"}
day3 = {"Priya", "Riya", "Meera", "Sanjay"}

perfect_attendance = day1 & day2 & day3
came_at_least_once = day1 | day2 | day3
never_came = class_roll - came_at_least_once
missed_day2 = class_roll - day2

print("Perfect attendance:   ", sorted(perfect_attendance))
print("Came at least once:   ", len(came_at_least_once), "of", len(class_roll))
print("Never came:           ", sorted(never_came))
print("Missed day 2:         ", sorted(missed_day2))

Perfect attendance:    ['Priya', 'Riya', 'Sanjay']
Came at least once:    5 of 6
Never came:            ['Kavita']
Missed day 2:          ['Kavita', 'Meera']


### Example 3: Delivery coverage by pincode

In [21]:
covered_pincodes = {400001, 400050, 400053, 400070, 411001, 411004}
requested_pincodes = {400050, 400053, 560001, 411004, 110001}

can_deliver = requested_pincodes & covered_pincodes
cannot_deliver = requested_pincodes - covered_pincodes

print("Can deliver to:    ", sorted(can_deliver))
print("Cannot deliver to: ", sorted(cannot_deliver))
print(f"Coverage of requests: {len(can_deliver)}/{len(requested_pincodes)}")

Can deliver to:     [400050, 400053, 411004]
Cannot deliver to:  [110001, 560001]
Coverage of requests: 3/5


### Example 4: Comparing two sentences

In [22]:
sentence_1 = "mumbai is a busy city and mumbai never sleeps"
sentence_2 = "delhi is a historic city and delhi never forgets"

words_1 = set(sentence_1.split())
words_2 = set(sentence_2.split())

print("Unique words in sentence 1:", len(words_1), "(of", len(sentence_1.split()), "total)")
print("Shared words:      ", sorted(words_1 & words_2))
print("Only in sentence 1:", sorted(words_1 - words_2))
print("Only in sentence 2:", sorted(words_2 - words_1))

Unique words in sentence 1: 8 (of 9 total)
Shared words:       ['a', 'and', 'city', 'is', 'never']
Only in sentence 1: ['busy', 'mumbai', 'sleeps']
Only in sentence 2: ['delhi', 'forgets', 'historic']


### Example 5: Blocklist filter

In [23]:
blocked_users = {"spam_bot_1", "fake_offer", "troll_99"}
comments = [
    ("priya_s", "Great video!"), ("fake_offer", "Win a free phone, click here"),
    ("arjun_m", "Please do a video on dictionaries"), ("troll_99", "boring"),
]

clean_comments = [(user, text) for user, text in comments if user not in blocked_users]
for user, text in clean_comments:
    print(f"{user}: {text}")
print("Removed:", len(comments) - len(clean_comments), "comments")

priya_s: Great video!
arjun_m: Please do a video on dictionaries
Removed: 2 comments


### Example 6: Matching candidates to a job (subset check)

A candidate qualifies when the required skills are a **subset** of their skills.

In [24]:
required_skills = {"python", "sql", "git"}

candidates = {
    "Priya":  {"python", "sql", "git", "docker"},
    "Arjun":  {"python", "git"},
    "Riya":   {"python", "sql", "git"},
    "Sanjay": {"excel", "sql"},
}

for name, skills in candidates.items():
    if required_skills <= skills:
        print(f"{name}: shortlisted")
    else:
        print(f"{name}: missing {sorted(required_skills - skills)}")

Priya: shortlisted
Arjun: missing ['sql']
Riya: shortlisted
Sanjay: missing ['git', 'python']


---
## Common mistakes

| Mistake | What happens | Fix |
|---------|--------------|-----|
| `empty = {}` | Creates a dict, not a set | Use `set()` |
| `my_set[0]` | `TypeError`: sets are not subscriptable | Use `sorted(my_set)[0]`, or a list if order matters |
| Putting a list in a set | `TypeError: unhashable type: 'list'` | Convert to a tuple first |
| `s.remove(x)` on a missing item | `KeyError` | Use `s.discard(x)` |
| Relying on `pop()` order | Removes an arbitrary item | Use a list if you need first/last |
| Expecting printed order to match insertion order | Order differs between runs | Use `sorted()` when you display |
| `a - b` vs `b - a` mixed up | Different results | Difference is not symmetric |
| `set(my_list)` when order matters | Order is lost | `list(dict.fromkeys(my_list))` |

## Predict the output

Try to answer each one **before** running the cell.

In [25]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(sorted(a & b))             # 1
print(sorted(a - b))             # 2
print(sorted(b - a))             # 3
print(len(a | b))                # 4
print(a.isdisjoint({7, 8}))      # 5
print(sorted(a ^ b))             # 6

[3, 4]
[1, 2]
[5, 6]
6
True
[1, 2, 5, 6]


**Answers:** 1) `[3, 4]`  2) `[1, 2]`  3) `[5, 6]`  4) `6`  5) `True`  6) `[1, 2, 5, 6]`

---
## Mini-challenges

### Challenge 1: Event invitations

Priya organised an event. Using set operations, find:
1. Invited guests who did **not** come
2. People who came **without** being invited
3. Everyone involved (invited or attended)

**Expected output:**
```
Did not come: ['Kavita', 'Sanjay']
Uninvited: ['Vikram']
Everyone involved: ['Arjun', 'Kavita', 'Meera', 'Riya', 'Sanjay', 'Vikram']
```

In [26]:
invited = {"Arjun", "Riya", "Sanjay", "Meera", "Kavita"}
attended = {"Arjun", "Riya", "Meera", "Vikram"}

# YOUR CODE HERE

In [ ]:
# SOLUTION:
# print("Did not come:", sorted(invited - attended))
# print("Uninvited:", sorted(attended - invited))
# print("Everyone involved:", sorted(invited | attended))

### Challenge 2: Unique hashtags

Each post has a list of hashtags. Build **one set** of all unique hashtags used across all posts using a set comprehension, then print how many there are and the hashtags that appear in **every** post.

**Expected output:**
```
Unique hashtags: 5
In every post: ['#python']
```

In [27]:
posts = [
    ["#python", "#coding", "#beginner"],
    ["#python", "#sets", "#coding"],
    ["#python", "#learning", "#sets"],
]

# YOUR CODE HERE

In [ ]:
# SOLUTION:
# all_tags = {tag for post in posts for tag in post}
# in_every_post = set(posts[0]).intersection(*posts[1:])
# print("Unique hashtags:", len(all_tags))
# print("In every post:", sorted(in_every_post))

### Challenge 3: Duplicate transaction IDs (harder)

A payment log (think Paytm or Razorpay) has repeated transaction IDs. Write code that prints the IDs that appear **more than once**, in sorted order, and the list of unique IDs **in their original order**.

**Expected output:**
```
Duplicate IDs: ['TXN102', 'TXN105']
Unique IDs in order: ['TXN101', 'TXN102', 'TXN103', 'TXN105', 'TXN104']
```

In [28]:
transaction_ids = ["TXN101", "TXN102", "TXN103", "TXN102", "TXN105", "TXN105", "TXN104", "TXN105"]

# YOUR CODE HERE

In [ ]:
# SOLUTION:
# seen = set()
# duplicates = set()
# for txn in transaction_ids:
#     if txn in seen:
#         duplicates.add(txn)
#     seen.add(txn)
# print("Duplicate IDs:", sorted(duplicates))
# print("Unique IDs in order:", list(dict.fromkeys(transaction_ids)))

---
## Summary

### Cheat sheet

| Task | Code |
|------|------|
| Create a set | `{1, 2, 3}` or `set(iterable)` |
| Empty set | `set()` (not `{}`) |
| Add one / many | `s.add(x)` / `s.update(items)` |
| Remove (strict / safe) | `s.remove(x)` / `s.discard(x)` |
| Membership | `x in s` |
| Union / intersection | `a \| b` / `a & b` |
| Difference / symmetric difference | `a - b` / `a ^ b` |
| Subset / superset / disjoint | `a <= b` / `a >= b` / `a.isdisjoint(b)` |
| Immutable set | `frozenset(items)` |
| Set comprehension | `{expr for x in items if cond}` |
| Unique items, order kept | `list(dict.fromkeys(items))` |
| Stable display order | `sorted(s)` |

### Key ideas

- A set stores **unique** items, with **no order**.
- Use `set()` for an empty set; `{}` makes a dictionary.
- Items must be **hashable**: tuples yes, lists no.
- Sets answer "is it in there?" and "what do these groups share?" faster and more clearly than lists.
- Difference is **not** symmetric: `a - b` is not `b - a`.
- Wrap in `sorted()` whenever you print a set and want a predictable order.